# Módulo 04 · Aula 06 — Concorrência

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O script consulta o CEP de 500 clientes na API dos Correios. Cada consulta leva 200 ms. Total: 100 segundos parado, esperando. A CPU fica em 2% o tempo todo."*

Concorrência resolve **este** problema — e **não** resolve outros. Esta aula é principalmente sobre saber a diferença.

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | I/O-bound vs CPU-bound | **A pergunta que define tudo** |
| 2 | O GIL | Por que threads não aceleram cálculo em Python |
| 3 | `threading` | Paralelismo de espera |
| 4 | `multiprocessing` | Paralelismo de cálculo |
| 5 | `concurrent.futures` | A interface unificada |
| 6 | **`asyncio`** | Milhares de esperas simultâneas |
| 7 | `async`/`await` | Sintaxe |
| 8 | Padrões e armadilhas | Race conditions, locks, cancelamento |

## 1. A pergunta que define tudo

> **Seu programa está esperando ou calculando?**

| | **I/O-bound** (esperando) | **CPU-bound** (calculando) |
|---|---------------------------|----------------------------|
| Gasta tempo em | Rede, disco, banco de dados | Contas, laços, processamento |
| CPU durante a espera | Ociosa (2%) | 100% |
| Exemplos | API, download, query SQL, leitura de arquivo | Ordenar 10M itens, criptografia, imagem |
| Solução | **threads** ou **asyncio** | **multiprocessing** |
| Ganho possível | 10× a 100× | No máximo = nº de núcleos |

**Se você errar o diagnóstico, a "otimização" deixa o programa mais lento.** Threads para trabalho de CPU adicionam sobrecarga sem ganho nenhum — por causa do GIL.

In [ ]:
import time
import threading
import multiprocessing
import os

print(f"Núcleos disponíveis: {os.cpu_count()}")
print(f"Python: {__import__('sys').version.split()[0]}")


def tarefa_io(duracao=0.1):
    """Simula espera de rede: NÃO usa CPU."""
    time.sleep(duracao)
    return duracao


def tarefa_cpu(n=2_000_000):
    """Cálculo puro: usa CPU 100%."""
    return sum(i * i for i in range(n))


t0 = time.perf_counter()
tarefa_io()
print(f"\ntarefa_io  : {(time.perf_counter()-t0)*1000:>7.0f} ms  (CPU ociosa)")

t0 = time.perf_counter()
tarefa_cpu()
print(f"tarefa_cpu : {(time.perf_counter()-t0)*1000:>7.0f} ms  (CPU a 100%)")

## 2. O GIL — *Global Interpreter Lock*

**O GIL é um cadeado que permite apenas uma thread executar bytecode Python por vez.**

```
Sem GIL (teoria)                  Com GIL (CPython)
─────────────────                 ─────────────────
Thread 1  ████████                Thread 1  ██··██··██
Thread 2  ████████                Thread 2  ··██··██··
Thread 3  ████████                Thread 3  ····██··██
          ↑ paralelo real                   ↑ alternado, nunca simultâneo
```

**Por que ele existe?** Para simplificar o gerenciamento de memória do CPython (contagem de referências). Removê-lo tornaria o Python de thread única mais lento e quebraria décadas de extensões em C.

**A parte crucial:** o GIL é **liberado durante operações de I/O**. Quando uma thread chama `time.sleep`, faz uma requisição HTTP ou lê um arquivo, ela solta o cadeado e outra thread trabalha.

| Trabalho | Threads ajudam? | Por quê |
|----------|:---------------:|---------|
| Esperar rede/disco/banco | ✅ **Sim** | O GIL é liberado durante a espera |
| Calcular em Python puro | ❌ **Não** | O GIL serializa a execução |
| NumPy/pandas pesado | ✅ Às vezes | Essas libs liberam o GIL no código C |

> ℹ️ **O GIL está sendo removido.** A [PEP 703](https://peps.python.org/pep-0703/) introduziu uma build experimental *free-threaded* no Python 3.13. Ainda é experimental e leva anos para virar padrão — mas o cenário vai mudar.

In [ ]:
# 🔬 EXPERIMENTO 1: CPU-bound com threads
def medir(descricao, funcao):
    inicio = time.perf_counter()
    funcao()
    duracao = time.perf_counter() - inicio
    print(f"{descricao:<34}{duracao:>7.2f}s")
    return duracao


N_TAREFAS = 4

def cpu_sequencial():
    for _ in range(N_TAREFAS):
        tarefa_cpu()

def cpu_com_threads():
    threads = [threading.Thread(target=tarefa_cpu) for _ in range(N_TAREFAS)]
    for t in threads: t.start()
    for t in threads: t.join()


print("CPU-BOUND (4 tarefas)")
print("─" * 42)
t_seq = medir("sequencial", cpu_sequencial)
t_thr = medir("com 4 threads", cpu_com_threads)
print("─" * 42)
print(f"{'ganho':<34}{t_seq/t_thr:>7.2f}x")
print("\n💭 Nenhum ganho — e às vezes fica PIOR, pela troca de contexto.")
print("   O GIL impede execução simultânea de bytecode Python.")

In [ ]:
# 🔬 EXPERIMENTO 2: I/O-bound com threads
def io_sequencial():
    for _ in range(N_TAREFAS):
        tarefa_io(0.3)

def io_com_threads():
    threads = [threading.Thread(target=tarefa_io, args=(0.3,)) for _ in range(N_TAREFAS)]
    for t in threads: t.start()
    for t in threads: t.join()


print("I/O-BOUND (4 tarefas de 300 ms)")
print("─" * 42)
t_seq = medir("sequencial", io_sequencial)
t_thr = medir("com 4 threads", io_com_threads)
print("─" * 42)
print(f"{'ganho':<34}{t_seq/t_thr:>7.2f}x")
print("\n💭 Ganho quase linear. As threads esperam JUNTAS.")
print("   O GIL foi liberado durante cada sleep.")

## 3. `threading`

In [ ]:
# Estado compartilhado sem proteção.
# Escrevemos o incremento nas TRÊS operações que ele realmente é.
def somar(vezes, n_threads, ceder=False, com_trava=False):
    contador = 0
    trava = threading.Lock()

    def incrementar():
        nonlocal contador
        for _ in range(vezes):
            if com_trava:
                with trava:
                    valor = contador              # 1. LÊ
                    if ceder: time.sleep(0)       #    (janela de troca)
                    contador = valor + 1          # 2. SOMA e 3. GRAVA
            else:
                valor = contador                  # 1. LÊ
                if ceder: time.sleep(0)           #    ⚠️ outra thread entra AQUI
                contador = valor + 1              # 2. SOMA e 3. GRAVA

    threads = [threading.Thread(target=incrementar) for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return contador


VEZES, N_THREADS = 20_000, 5
ESPERADO = VEZES * N_THREADS

obtido = somar(VEZES, N_THREADS)
print(f"Esperado : {ESPERADO:,}")
print(f"Obtido   : {obtido:,}")
print(f"Perdidos : {ESPERADO - obtido:,}")

> 🤨 **Deu certo? Provavelmente sim — e é exatamente esse o problema.**
>
> O CPython troca de thread a cada ~5 milissegundos. Nesse intervalo, uma thread executa milhares de iterações inteiras sem interrupção. A chance de a troca cair **exatamente** entre o "ler" e o "gravar" é pequena.
>
> **Mas o bug está lá.** Ele só não se manifestou.
>
> É assim que race conditions se comportam: passam em todos os testes da sua máquina, e quebram em produção, sob carga, uma vez a cada dez mil execuções — sem se reproduzir quando você tenta investigar.
>
> Vamos forçar a troca a acontecer no ponto perigoso, com um `time.sleep(0)` entre a leitura e a gravação. O `sleep(0)` não dorme: ele apenas **cede o GIL**.

In [ ]:
print(f"{'cenário':<34}{'obtido':>10}{'perdidos':>12}{'':>4}")
print("─" * 60)

for ceder, trava, rotulo in [
    (False, False, "sem ceder, sem trava"),
    (True,  False, "cedendo o GIL, sem trava"),
    (True,  True,  "cedendo o GIL, COM trava"),
]:
    obtido = somar(VEZES, N_THREADS, ceder=ceder, com_trava=trava)
    perdidos = ESPERADO - obtido
    marca = "✅" if perdidos == 0 else "🔴"
    print(f"{rotulo:<34}{obtido:>10,}{perdidos:>12,}  {marca}")

print("─" * 60)
print(f"{'esperado':<34}{ESPERADO:>10,}")

> 🔴 **Ali está: 80% dos incrementos evaporaram.**
>
> **Por que acontece:** `contador += 1` são três operações. Duas threads leem o valor `100` ao mesmo tempo, ambas somam 1, ambas gravam `101`. Dois incrementos viraram um.
>
> ```
> Thread A:  lê 100 ─────────────► soma → 101 ──► grava 101
> Thread B:       lê 100 ──► soma → 101 ──► grava 101
>                                                   ↑
>                                        um incremento se perdeu
> ```
>
> **E na terceira linha, o `Lock` resolveu** — ele torna as três operações indivisíveis.
>
> 💭 **A lição não é "use Lock".** É que **você não controla quando a troca acontece**. Aqui forçamos com `sleep(0)`; em produção, a troca vem naturalmente de uma chamada de rede, do coletor de lixo ou da concorrência por CPU.
>
> **Seu teste passar não prova que o código é seguro** — prova apenas que a troca não caiu no lugar errado *daquela vez*.

In [ ]:
# O custo do Lock
inicio = time.perf_counter()
somar(VEZES, N_THREADS, com_trava=False)
t_sem = time.perf_counter() - inicio

inicio = time.perf_counter()
somar(VEZES, N_THREADS, com_trava=True)
t_com = time.perf_counter() - inicio

print(f"sem trava : {t_sem*1000:>7.0f} ms   (e potencialmente errado)")
print(f"com trava : {t_com*1000:>7.0f} ms   ({t_com/t_sem:.1f}x mais lento)")
print("\n💭 Correção custa desempenho. Resultado errado custa mais.")

> 🧭 **A melhor solução para race condition é não ter estado compartilhado.**
>
> Em vez de várias threads escrevendo na mesma variável, faça cada uma devolver seu resultado e agregue no final. É mais simples, mais rápido e impossível de errar. É exatamente o que `concurrent.futures` facilita.
>
> Quando o compartilhamento é inevitável, prefira `queue.Queue`, que já é thread-safe.

In [ ]:
# Ferramentas de sincronização
import queue

fila = queue.Queue()
resultados = []
trava_resultado = threading.Lock()


def trabalhador(nome):
    while True:
        item = fila.get()
        if item is None:               # sentinela de parada
            fila.task_done()
            break
        time.sleep(0.05)               # simula processamento
        with trava_resultado:
            resultados.append((nome, item, item ** 2))
        fila.task_done()


trabalhadores = [threading.Thread(target=trabalhador, args=(f"w{i}",), daemon=True)
                 for i in range(3)]
for t in trabalhadores:
    t.start()

for n in range(1, 10):
    fila.put(n)
for _ in trabalhadores:
    fila.put(None)                     # uma sentinela por trabalhador

fila.join()

print(f"{len(resultados)} itens processados por 3 trabalhadores:")
from collections import Counter
print("Distribuição:", dict(Counter(r[0] for r in resultados)))

## 4. `multiprocessing` — paralelismo real

Cada processo tem **seu próprio interpretador e sua própria memória**. Portanto, **seu próprio GIL**.

| | Thread | Processo |
|---|--------|----------|
| Memória | Compartilhada | Isolada |
| Custo de criação | Baixo | Alto (~10-50 ms) |
| Comunicação | Direta (e perigosa) | Serialização (`pickle`) |
| Afetado pelo GIL | ✅ Sim | ❌ Não |
| Para que serve | I/O | CPU |

> ⚠️ **`multiprocessing` não funciona bem em notebooks.** As funções precisam ser importáveis de um módulo (o `pickle` não serializa funções definidas no `__main__` interativo). Por isso vamos escrever um arquivo `.py`.
>
> ⚠️ **No Windows e macOS**, o método de início é `spawn`: o processo filho **reimporta** o módulo. Sem `if __name__ == "__main__":`, você cria processos infinitamente. Este é o motivo mais concreto de a aula 01_04 ter insistido nesse bloco.

In [ ]:
%%writefile tarefas_paralelas.py
"""Funções para multiprocessing.

⚠️ Precisam estar em um MÓDULO (não no notebook) para que o pickle
   consiga serializá-las e enviá-las aos processos filhos.
"""

import time


def cpu_pesado(n: int) -> int:
    """Trabalho de CPU puro."""
    return sum(i * i for i in range(n))


def io_lento(duracao: float) -> float:
    """Trabalho de espera."""
    time.sleep(duracao)
    return duracao


def processar_lote(lote: list[dict]) -> dict:
    """Agrega um lote de vendas — o caso real do Atlas."""
    total, itens = 0.0, 0
    for venda in lote:
        total += venda["quantidade"] * venda["preco"]
        itens += venda["quantidade"]
    return {"registros": len(lote), "itens": itens, "receita": round(total, 2)}


if __name__ == "__main__":
    print(cpu_pesado(1000))

In [ ]:
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import tarefas_paralelas as tp

N = 2_000_000
TAREFAS = [N] * 4

# Sequencial
inicio = time.perf_counter()
r_seq = [tp.cpu_pesado(n) for n in TAREFAS]
t_seq = time.perf_counter() - inicio

# Com processos
inicio = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    r_proc = list(executor.map(tp.cpu_pesado, TAREFAS))
t_proc = time.perf_counter() - inicio

# Com threads (para comparar)
inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    r_thr = list(executor.map(tp.cpu_pesado, TAREFAS))
t_thr = time.perf_counter() - inicio

print("CPU-BOUND — 4 tarefas")
print("─" * 46)
print(f"{'sequencial':<24}{t_seq:>8.2f}s{'—':>10}")
print(f"{'ThreadPoolExecutor':<24}{t_thr:>8.2f}s{t_seq/t_thr:>9.2f}x")
print(f"{'ProcessPoolExecutor':<24}{t_proc:>8.2f}s{t_seq/t_proc:>9.2f}x")
print("─" * 46)
print(f"Resultados idênticos? {r_seq == r_proc == r_thr}")
print("\n💭 Só os PROCESSOS aceleram trabalho de CPU.")

## 5. `concurrent.futures` — a interface unificada

Esta é a API que você deve usar no dia a dia. Ela abstrai threads e processos atrás da mesma interface: **troque uma palavra e mude a estratégia**.

| Classe | Usa | Para |
|--------|-----|------|
| `ThreadPoolExecutor` | Threads | I/O-bound |
| `ProcessPoolExecutor` | Processos | CPU-bound |

| Método | Faz |
|--------|-----|
| `submit(f, *args)` | Agenda uma tarefa, devolve um `Future` |
| `map(f, iteravel)` | Como `map`, em paralelo, **preserva a ordem** |
| `as_completed(futures)` | Itera **conforme terminam** |
| `future.result()` | Espera e devolve (ou **relança a exceção**) |

In [ ]:
from concurrent.futures import as_completed
import random

rnd = random.Random(42)

def consultar_cep(cep: str) -> dict:
    """Simula consulta a uma API externa, com latência variável."""
    latencia = rnd.uniform(0.05, 0.25)
    time.sleep(latencia)
    if cep.endswith("999"):
        raise ConnectionError(f"timeout ao consultar {cep}")
    return {"cep": cep, "cidade": "Campinas", "uf": "SP", "latencia_ms": round(latencia*1000)}


ceps = [f"1300{i:04d}" for i in range(20)] + ["13009999"]

# Sequencial
inicio = time.perf_counter()
sequencial = []
for cep in ceps:
    try:
        sequencial.append(consultar_cep(cep))
    except ConnectionError:
        pass
t_seq = time.perf_counter() - inicio
print(f"Sequencial : {t_seq:.2f}s  ({len(sequencial)} sucessos)")

In [ ]:
# Com ThreadPoolExecutor
inicio = time.perf_counter()
sucessos, falhas = [], []

with ThreadPoolExecutor(max_workers=10) as executor:
    futuros = {executor.submit(consultar_cep, cep): cep for cep in ceps}
    for futuro in as_completed(futuros):
        cep = futuros[futuro]
        try:
            sucessos.append(futuro.result())
        except ConnectionError as erro:
            falhas.append((cep, str(erro)))

t_par = time.perf_counter() - inicio

print(f"Paralelo   : {t_par:.2f}s  ({len(sucessos)} sucessos, {len(falhas)} falhas)")
print(f"Ganho      : {t_seq/t_par:.1f}x")
print(f"\nFalhas tratadas individualmente: {falhas}")

> 💡 **`as_completed` vs `map`:**
>
> - **`map`** preserva a ordem da entrada e é mais simples. Se a 1ª tarefa demora 10s e as outras 10ms, você espera 10s pelo primeiro resultado.
> - **`as_completed`** entrega conforme terminam. Melhor quando você quer processar resultados incrementalmente ou mostrar progresso.
>
> ⚠️ **A exceção fica guardada no `Future`** e só é relançada quando você chama `.result()`. Se você nunca chamar, o erro **desaparece em silêncio**. Sempre trate.

In [ ]:
# Escolhendo o executor conforme o tipo de trabalho
def comparar(descricao, funcao, argumentos, n_workers=4):
    inicio = time.perf_counter()
    resultado = [funcao(a) for a in argumentos]
    t_seq = time.perf_counter() - inicio

    inicio = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        list(ex.map(funcao, argumentos))
    t_thr = time.perf_counter() - inicio

    inicio = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        list(ex.map(funcao, argumentos))
    t_proc = time.perf_counter() - inicio

    print(f"\n{descricao}")
    print("─" * 48)
    print(f"{'sequencial':<20}{t_seq:>9.2f}s")
    print(f"{'threads':<20}{t_thr:>9.2f}s  ({t_seq/t_thr:>5.2f}x)")
    print(f"{'processos':<20}{t_proc:>9.2f}s  ({t_seq/t_proc:>5.2f}x)")
    melhor = min([(t_seq, "sequencial"), (t_thr, "threads"), (t_proc, "processos")])
    print(f"→ melhor: {melhor[1]}")


comparar("I/O-BOUND (4 × 400ms de espera)", tp.io_lento, [0.4] * 4)
comparar("CPU-BOUND (4 × 2M operações)", tp.cpu_pesado, [2_000_000] * 4)

> 💭 **Repare no custo dos processos no caso I/O.** Criar 4 processos leva mais tempo que a própria espera. Processos têm sobrecarga de inicialização e de serialização — só compensam quando o trabalho é pesado o suficiente para pagar por ela.

## 6. `asyncio` — concorrência sem threads

O `asyncio` faz concorrência de I/O **em uma única thread**, usando um *event loop* e **multitarefa cooperativa**.

```
THREADS                          ASYNCIO
───────                          ───────
N threads do sistema             1 thread
Troca PREEMPTIVA (o SO decide)   Troca COOPERATIVA (no await)
~8 MB de pilha cada              ~KB por tarefa
Milhares = problema              Milhares = tranquilo
Race conditions possíveis        Pontos de troca são visíveis (await)
```

**A regra da multitarefa cooperativa:** uma corrotina só cede o controle quando chega em um `await`. Se ela nunca faz `await`, ela **trava o loop inteiro**.

| Conceito | O que é |
|----------|---------|
| `async def` | Define uma **corrotina** |
| `await` | Cede o controle enquanto espera |
| Event loop | Quem decide qual corrotina roda |
| `Task` | Corrotina agendada para execução |
| `asyncio.run()` | Inicia o loop e roda tudo |
| `asyncio.gather()` | Executa várias e espera todas |

In [ ]:
import asyncio

async def consultar_async(cep: str, latencia: float) -> dict:
    """Corrotina: note o async def e o await."""
    await asyncio.sleep(latencia)      # ⚠️ asyncio.sleep, NÃO time.sleep
    return {"cep": cep, "latencia_ms": round(latencia * 1000)}


async def principal():
    inicio = time.perf_counter()

    # ❌ SEQUENCIAL: await um de cada vez
    r1 = await consultar_async("13001000", 0.3)
    r2 = await consultar_async("13002000", 0.3)
    r3 = await consultar_async("13003000", 0.3)
    t_seq = time.perf_counter() - inicio

    # ✅ CONCORRENTE: gather dispara todas juntas
    inicio = time.perf_counter()
    resultados = await asyncio.gather(
        consultar_async("13001000", 0.3),
        consultar_async("13002000", 0.3),
        consultar_async("13003000", 0.3),
    )
    t_par = time.perf_counter() - inicio

    print(f"3 esperas de 300 ms:")
    print(f"  await sequencial : {t_seq:.2f}s")
    print(f"  asyncio.gather   : {t_par:.2f}s")
    print(f"  ganho            : {t_seq/t_par:.1f}x")
    return resultados


# Em notebook o loop JÁ ESTÁ RODANDO — não se pode chamar asyncio.run().
# Nesta simulação usamos asyncio.run(); em Jupyter de verdade, use `await principal()`
# direto na célula (o IPython suporta await de nível superior).
resultados = asyncio.run(principal())
print(f"\n{len(resultados)} resultados")

> ⚠️ **No Jupyter, o event loop já está rodando.** Isso significa:
>
> - `asyncio.run(...)` levanta `RuntimeError: This event loop is already running`
> - Mas o IPython permite **`await` direto na célula**: basta escrever `resultados = await principal()`
>
> Nas células a seguir usamos `asyncio.run` dentro de uma função auxiliar para funcionar em qualquer contexto. Ao praticar no seu notebook, prefira o `await` direto — é mais simples.

In [ ]:
# Escala: 200 requisições simultâneas
async def buscar(id_: int) -> dict:
    await asyncio.sleep(rnd.uniform(0.05, 0.2))
    if id_ % 37 == 0:
        raise ConnectionError(f"falha no id {id_}")
    return {"id": id_, "ok": True}


async def buscar_todos(n: int):
    inicio = time.perf_counter()
    # return_exceptions=True: as exceções voltam como VALORES, não abortam
    resultados = await asyncio.gather(*(buscar(i) for i in range(n)),
                                      return_exceptions=True)
    duracao = time.perf_counter() - inicio

    sucessos = [r for r in resultados if not isinstance(r, Exception)]
    falhas = [r for r in resultados if isinstance(r, Exception)]
    return duracao, sucessos, falhas


duracao, sucessos, falhas = asyncio.run(buscar_todos(200))

print(f"200 requisições de ~125 ms cada")
print(f"  sequencial seria : ~{200 * 0.125:.0f}s")
print(f"  com asyncio      : {duracao:.2f}s")
print(f"  ganho            : ~{200 * 0.125 / duracao:.0f}x")
print(f"\n  sucessos: {len(sucessos)} | falhas: {len(falhas)}")
print(f"  exemplo de falha: {falhas[0]}")

> 💡 **`return_exceptions=True` é quase sempre o que você quer.** Sem ele, a primeira exceção aborta o `gather` e você perde os resultados das outras 199 tarefas — que já tinham dado certo.

In [ ]:
# Limitando a concorrência: Semaphore
async def buscar_com_limite(id_: int, semaforo: asyncio.Semaphore) -> int:
    async with semaforo:               # no máximo N aqui dentro
        await asyncio.sleep(0.05)
        return id_


async def com_semaforo(n_tarefas, limite):
    semaforo = asyncio.Semaphore(limite)
    inicio = time.perf_counter()
    await asyncio.gather(*(buscar_com_limite(i, semaforo) for i in range(n_tarefas)))
    return time.perf_counter() - inicio


for limite in [5, 20, 100]:
    t = asyncio.run(com_semaforo(100, limite))
    print(f"100 tarefas, limite {limite:>3}: {t:.2f}s")

print("\n💭 Por que limitar? Porque a API do outro lado tem rate limit,")
print("   e 500 requisições simultâneas geram 429 (Too Many Requests)")
print("   ou derrubam o serviço. Ser educado é parte do trabalho.")

### 🔴 A armadilha: código bloqueante dentro de async

Se você chamar `time.sleep()` (ou qualquer função síncrona lenta) dentro de uma corrotina, **o event loop inteiro trava**. Nenhuma outra tarefa roda.

In [ ]:
async def bloqueante(id_):
    time.sleep(0.2)                    # 🔴 BLOQUEIA O LOOP
    return id_


async def nao_bloqueante(id_):
    await asyncio.sleep(0.2)           # ✅ cede o controle
    return id_


async def comparar_bloqueio():
    inicio = time.perf_counter()
    await asyncio.gather(*(bloqueante(i) for i in range(5)))
    t_bloq = time.perf_counter() - inicio

    inicio = time.perf_counter()
    await asyncio.gather(*(nao_bloqueante(i) for i in range(5)))
    t_ok = time.perf_counter() - inicio

    return t_bloq, t_ok


t_bloq, t_ok = asyncio.run(comparar_bloqueio())
print(f"5 tarefas de 200 ms:")
print(f"  com time.sleep    : {t_bloq:.2f}s  ← serializou, sem ganho nenhum")
print(f"  com asyncio.sleep : {t_ok:.2f}s  ← concorrente")

> 🧭 **A regra:** em código `async`, use bibliotecas `async`.
>
> | Síncrono (bloqueia) | Assíncrono |
> |---------------------|------------|
> | `time.sleep` | `asyncio.sleep` |
> | `requests` | `httpx` (async) ou `aiohttp` |
> | `open()` | `aiofiles` |
> | `psycopg` | `asyncpg` |
> | `sqlite3` | `aiosqlite` |
>
> **E se você PRECISA chamar código síncrono?** Use `asyncio.to_thread(funcao, *args)` — ele roda a função numa thread separada sem travar o loop.

In [ ]:
async def com_to_thread():
    """Executando código bloqueante sem travar o loop."""
    inicio = time.perf_counter()
    resultados = await asyncio.gather(
        *(asyncio.to_thread(time.sleep, 0.2) for _ in range(5))
    )
    return time.perf_counter() - inicio


t = asyncio.run(com_to_thread())
print(f"5 × time.sleep(0.2) via to_thread: {t:.2f}s  ✅ concorrente de novo")

In [ ]:
# Timeout e cancelamento
async def operacao_lenta(segundos):
    try:
        await asyncio.sleep(segundos)
        return "concluído"
    except asyncio.CancelledError:
        print("   ↩️  operação cancelada — limpando recursos")
        raise


async def com_timeout():
    print("① Dentro do prazo:")
    try:
        r = await asyncio.wait_for(operacao_lenta(0.1), timeout=0.5)
        print(f"   ✅ {r}")
    except asyncio.TimeoutError:
        print("   ⏱ timeout")

    print("\n② Estourando o prazo:")
    try:
        r = await asyncio.wait_for(operacao_lenta(2.0), timeout=0.3)
        print(f"   ✅ {r}")
    except asyncio.TimeoutError:
        print("   ⏱ timeout — a tarefa foi cancelada automaticamente")


asyncio.run(com_timeout())

## 7. Comparação final

| Critério | Threads | Processos | asyncio |
|----------|---------|-----------|---------|
| Melhor para | I/O moderado | CPU | I/O em massa |
| Custo por unidade | ~8 MB | ~10-50 MB | ~KB |
| Escala confortável | dezenas | nº de núcleos | **milhares** |
| Afetado pelo GIL | Sim | Não | Sim (irrelevante para I/O) |
| Race conditions | ✅ possíveis | ❌ (memória isolada) | Raras (troca só no `await`) |
| Complexidade | Média | Média | **Alta** (contamina toda a pilha) |
| Precisa reescrever o código | Não | Não | **Sim** — tudo vira `async` |

### Como decidir

```
O gargalo é ESPERA (rede/disco/banco)?
├── Sim
│   ├── Dezenas de operações?      → ThreadPoolExecutor  (simples)
│   └── Centenas ou milhares?      → asyncio             (escala)
└── Não (é CÁLCULO)
    ├── Python puro?               → ProcessPoolExecutor
    └── NumPy/pandas?              → já é paralelo em C; meça antes
```

> 🧭 **O conselho mais importante: MEÇA ANTES.**
>
> Concorrência adiciona complexidade, bugs difíceis de reproduzir e dor de depuração. Se o programa leva 2 segundos, não vale a pena. **Otimize o algoritmo primeiro** — trocar um `list` por `set` numa busca costuma render mais que qualquer paralelismo.

## 🔧 Prática guiada — Enriquecendo dados da Aurora

Cenário real: 300 clientes sem UF preenchida. A API de CEP responde em ~150 ms. Vamos comparar as quatro abordagens.

In [ ]:
%%writefile enriquecimento.py
"""Enriquecimento de clientes via API de CEP — as 4 abordagens."""

import asyncio
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

_rnd = random.Random(42)

UFS = ["SP", "RJ", "MG", "PR", "SC", "RS", "BA", "PE"]
CIDADES = {"SP": "Campinas", "RJ": "Niterói", "MG": "Uberlândia", "PR": "Londrina",
           "SC": "Joinville", "RS": "Pelotas", "BA": "Feira de Santana", "PE": "Olinda"}


def _resposta(cep: str) -> dict:
    """Resposta determinística a partir do CEP."""
    indice = int(cep[-3:]) % len(UFS)
    uf = UFS[indice]
    return {"cep": cep, "uf": uf, "cidade": CIDADES[uf]}


def consultar_sincrono(cep: str, latencia: float = 0.15) -> dict:
    """Versão bloqueante — simula requests.get()."""
    time.sleep(latencia)
    if cep.endswith("00"):
        raise ConnectionError(f"timeout: {cep}")
    return _resposta(cep)


async def consultar_async(cep: str, latencia: float = 0.15) -> dict:
    """Versão assíncrona — simula httpx.AsyncClient.get()."""
    await asyncio.sleep(latencia)
    if cep.endswith("00"):
        raise ConnectionError(f"timeout: {cep}")
    return _resposta(cep)


# ── Estratégia 1: sequencial ──
def enriquecer_sequencial(ceps):
    ok, erro = [], []
    for cep in ceps:
        try:
            ok.append(consultar_sincrono(cep))
        except ConnectionError as e:
            erro.append(str(e))
    return ok, erro


# ── Estratégia 2: threads ──
def enriquecer_threads(ceps, workers=20):
    ok, erro = [], []
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futuros = [executor.submit(consultar_sincrono, c) for c in ceps]
        for f in as_completed(futuros):
            try:
                ok.append(f.result())
            except ConnectionError as e:
                erro.append(str(e))
    return ok, erro


# ── Estratégia 3: asyncio ──
async def enriquecer_async(ceps, limite=50):
    semaforo = asyncio.Semaphore(limite)

    async def com_limite(cep):
        async with semaforo:
            return await consultar_async(cep)

    resultados = await asyncio.gather(*(com_limite(c) for c in ceps),
                                      return_exceptions=True)
    ok = [r for r in resultados if not isinstance(r, Exception)]
    erro = [str(r) for r in resultados if isinstance(r, Exception)]
    return ok, erro


# ── Estratégia 4: asyncio com retentativa ──
async def consultar_com_retry(cep, tentativas=3, espera_base=0.05):
    """Retentativa com backoff exponencial — o padrão de produção."""
    for tentativa in range(1, tentativas + 1):
        try:
            return await consultar_async(cep)
        except ConnectionError:
            if tentativa == tentativas:
                raise
            await asyncio.sleep(espera_base * (2 ** (tentativa - 1)))


async def enriquecer_async_resiliente(ceps, limite=50):
    semaforo = asyncio.Semaphore(limite)

    async def com_limite(cep):
        async with semaforo:
            return await consultar_com_retry(cep)

    resultados = await asyncio.gather(*(com_limite(c) for c in ceps),
                                      return_exceptions=True)
    ok = [r for r in resultados if not isinstance(r, Exception)]
    erro = [str(r) for r in resultados if isinstance(r, Exception)]
    return ok, erro

In [ ]:
import importlib
import enriquecimento as enr
importlib.reload(enr)
import asyncio, time

CEPS = [f"13{i:06d}" for i in range(1, 301)]
print(f"{len(CEPS)} CEPs a consultar, ~150 ms cada")
print(f"Tempo teórico sequencial: {len(CEPS) * 0.15:.0f}s\n")

# Sequencial só com uma amostra — 300 × 150ms seriam 45 segundos
amostra = CEPS[:20]
inicio = time.perf_counter()
ok, erro = enr.enriquecer_sequencial(amostra)
t_amostra = time.perf_counter() - inicio
t_seq_estimado = t_amostra / len(amostra) * len(CEPS)

print(f"{'SEQUENCIAL (20 de 300)':<32}{t_amostra:>8.2f}s")
print(f"{'  → estimativa para 300':<32}{t_seq_estimado:>8.2f}s")

In [ ]:
inicio = time.perf_counter()
ok_thr, erro_thr = enr.enriquecer_threads(CEPS, workers=30)
t_thr = time.perf_counter() - inicio
print(f"{'THREADS (30 workers)':<32}{t_thr:>8.2f}s   "
      f"({len(ok_thr)} ok, {len(erro_thr)} falhas)")

inicio = time.perf_counter()
ok_asy, erro_asy = asyncio.run(enr.enriquecer_async(CEPS, limite=50))
t_asy = time.perf_counter() - inicio
print(f"{'ASYNCIO (limite 50)':<32}{t_asy:>8.2f}s   "
      f"({len(ok_asy)} ok, {len(erro_asy)} falhas)")

inicio = time.perf_counter()
ok_res, erro_res = asyncio.run(enr.enriquecer_async_resiliente(CEPS, limite=50))
t_res = time.perf_counter() - inicio
print(f"{'ASYNCIO + retry':<32}{t_res:>8.2f}s   "
      f"({len(ok_res)} ok, {len(erro_res)} falhas)")

In [ ]:
print("RESUMO")
print("─" * 58)
print(f"{'Estratégia':<26}{'Tempo':>10}{'Ganho':>10}{'Sucessos':>12}")
print("─" * 58)
print(f"{'sequencial (estimado)':<26}{t_seq_estimado:>9.2f}s{'—':>10}{'—':>12}")
print(f"{'threads':<26}{t_thr:>9.2f}s{t_seq_estimado/t_thr:>9.1f}x{len(ok_thr):>12}")
print(f"{'asyncio':<26}{t_asy:>9.2f}s{t_seq_estimado/t_asy:>9.1f}x{len(ok_asy):>12}")
print(f"{'asyncio + retry':<26}{t_res:>9.2f}s{t_seq_estimado/t_res:>9.1f}x{len(ok_res):>12}")
print("─" * 58)

print("\n💭 A retentativa é mais LENTA e recupera MAIS registros.")
print("   Esse é o trade-off real: tempo contra completude.")
print("   Qual importa mais depende do que o dado alimenta.")

In [ ]:
# O resultado do enriquecimento
from collections import Counter

distribuicao = Counter(r["uf"] for r in ok_res)
print("Clientes enriquecidos por UF:")
for uf, n in distribuicao.most_common():
    print(f"  {uf}  {n:>4}  {'█' * (n // 2)}")

print(f"\nTotal enriquecido: {len(ok_res)}/{len(CEPS)} "
      f"({len(ok_res)/len(CEPS):.1%})")
if erro_res:
    print(f"Não resolvidos   : {len(erro_res)} — exemplo: {erro_res[0]}")

> 💭 **Volte à dor do início.** *"500 consultas × 200 ms = 100 segundos parado."*
>
> Com `asyncio` e limite de 50 simultâneas, isso vira poucos segundos — e a CPU continua em 2%, porque o trabalho sempre foi de **espera**, não de cálculo.
>
> No **Módulo 07**, essas funções viram integrações reais com `httpx`, e o padrão de retentativa com backoff ganha a biblioteca `tenacity`.

## 📝 Exercícios

**E1.** Classifique cada tarefa como I/O-bound ou CPU-bound e diga qual ferramenta usaria: (a) baixar 1.000 imagens; (b) redimensionar 1.000 imagens; (c) ler 50 CSVs do disco; (d) calcular hash SHA-256 de 50 arquivos; (e) 200 consultas ao PostgreSQL; (f) ordenar 10 milhões de registros.

**E2.** Demonstre o GIL: meça uma tarefa CPU-bound com 1, 2, 4 e 8 threads. Faça um gráfico ASCII dos tempos.

**E3.** Repita o E2 com processos e compare as duas curvas.

**E4.** Reproduza uma race condition com uma lista compartilhada. Corrija com `Lock` e depois com `queue.Queue`. Compare a performance das duas correções.

**E5.** Implemente um pool de trabalhadores com `queue.Queue` que processe uma fila de tarefas com N threads e colete os resultados de forma segura.

**E6.** Use `ProcessPoolExecutor` para agregar 1 milhão de vendas em 8 lotes paralelos. Combine os resultados parciais. Compare com a versão sequencial.

**E7.** Escreva uma função que aceite `estrategia="sequencial"|"threads"|"processos"` e escolha o executor. Meça as três em I/O e em CPU.

**E8.** Converta uma função síncrona de download em `async`. Compare `gather` com e sem `Semaphore` de limite 10.

**E9.** Implemente `asyncio.gather` com barra de progresso usando `asyncio.as_completed`.

**E10.** Escreva `retry_async(funcao, tentativas, backoff)` com backoff exponencial **e jitter** (aleatoriedade para evitar sincronização de retentativas).

**E11.** Use `asyncio.wait_for` para impor timeout individual por tarefa, e reporte quantas estouraram.

**E12.** Demonstre a armadilha do bloqueio: rode 10 corrotinas com `time.sleep` e depois com `asyncio.to_thread`. Explique a diferença.

**E13.** Implemente um *rate limiter* assíncrono que garanta no máximo N requisições por segundo (não apenas N simultâneas).

**E14.** Combine `asyncio` (para buscar dados de API) com `ProcessPoolExecutor` (para processá-los), usando `loop.run_in_executor`.

**E15.** Meça o custo de criação: 1.000 threads vs 1.000 processos vs 1.000 tarefas asyncio. Reporte tempo e memória.

In [ ]:
# E1 — sua tabela de classificação

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

## 📋 Cola de referência

```python
# ── A pergunta ──
# Esperando (rede/disco/banco)? → threads ou asyncio
# Calculando (Python puro)?     → processos

# ── GIL ──
# Uma thread executa bytecode por vez.
# LIBERADO durante I/O → threads ajudam em espera, não em cálculo.

# ── threading ──
import threading
t = threading.Thread(target=funcao, args=(a, b), daemon=True)
t.start();  t.join()

trava = threading.Lock()
with trava:
    estado_compartilhado += 1        # ⚠️ += NÃO é atômico

import queue
fila = queue.Queue()                 # thread-safe por natureza
fila.put(x);  fila.get();  fila.task_done();  fila.join()

# ── concurrent.futures (use esta API) ──
from concurrent.futures import (ThreadPoolExecutor, ProcessPoolExecutor,
                                as_completed)

with ThreadPoolExecutor(max_workers=10) as ex:      # I/O
    resultados = list(ex.map(funcao, itens))        # preserva a ordem

    futuros = {ex.submit(f, x): x for x in itens}
    for fut in as_completed(futuros):               # conforme terminam
        try:
            r = fut.result()                        # ⚠️ relança a exceção aqui
        except Exception as e:
            ...

with ProcessPoolExecutor() as ex:                   # CPU
    ...
# ⚠️ funções precisam ser importáveis de um módulo
# ⚠️ exige if __name__ == "__main__" no script

# ── asyncio ──
import asyncio

async def corrotina(x):
    await asyncio.sleep(1)           # cede o controle
    return x

asyncio.run(principal())             # script
await principal()                    # ✅ dentro do Jupyter

await asyncio.gather(*tarefas, return_exceptions=True)   # ✅ sempre
async with asyncio.Semaphore(10):    # limita simultaneidade
await asyncio.wait_for(coro, timeout=5.0)
await asyncio.to_thread(funcao_sincrona, arg)           # sem travar o loop
task = asyncio.create_task(coro);  task.cancel()

for fut in asyncio.as_completed(tarefas):
    r = await fut

# 🔴 NUNCA dentro de async: time.sleep, requests, open()
#    Use: asyncio.sleep, httpx/aiohttp, aiofiles

# ── Decisão ──
# dezenas de I/O      → ThreadPoolExecutor  (simples)
# milhares de I/O     → asyncio             (escala)
# CPU em Python puro  → ProcessPoolExecutor
# 🧭 MEÇA ANTES. Otimize o algoritmo primeiro.
```

## ✅ Checklist de saída

- [ ] **Faço a pergunta certa: está esperando ou calculando?**
- [ ] Explico o que é o GIL e por que ele existe
- [ ] Sei que o GIL é liberado durante I/O
- [ ] Sei que threads não aceleram CPU-bound em Python
- [ ] Reconheço uma race condition e sei corrigi-la
- [ ] Prefiro evitar estado compartilhado a proteger com `Lock`
- [ ] Sei que processos têm memória isolada e custo de serialização
- [ ] Sei que `multiprocessing` exige funções em módulo e `__main__`
- [ ] **Uso `concurrent.futures` como interface padrão**
- [ ] Diferencio `map` de `as_completed`
- [ ] Sempre trato exceções ao chamar `future.result()`
- [ ] Entendo multitarefa cooperativa e o papel do `await`
- [ ] Uso `gather(..., return_exceptions=True)`
- [ ] Limito simultaneidade com `Semaphore`
- [ ] **Nunca chamo função bloqueante dentro de corrotina**
- [ ] Conheço `asyncio.to_thread` para código síncrono legado
- [ ] Uso `wait_for` para timeout
- [ ] **Meço antes de paralelizar**

---

### ➡️ Próxima etapa

**`04_99_Lista_Exercicios.ipynb`** — Lista completa do módulo e o projeto: refatorar o Atlas para orientação a objetos com logging estruturado.